<img src="https://sfile.chatglm.cn/images-ppt/rnn_lstm_cover.png" width="500" align="center">

# Рекуррентные сети (RNN/LSTM) - как нейросети "запоминают"

**Роль:** Преподаватель по ML
**Уровень:** средний (основы PyTorch + обработка текста)
**Время прохождения:** ~90-120 минут

---

### Чему вы научитесь

После прохождения этого ноутбука вы:
- поймёте, **зачем** нужна рекуррентность в нейросетях;
- увидите, как работает **простая RNN** - шаг за шагом;
- разберёте проблему **исчезающего градиента** визуально;
- реализуете **LSTM** с нуля и поймёте каждый гейт;
- **обучите** модель для классификации текста;
- визуализируете **скрытые состояния** и поймёте, что запоминает RNN;
- сравните RNN, LSTM и GRU на практике.

### Принцип этого блокнота

> Вы - автор, не пользователь. Каждая строка видна. Каждое действие можно сломать и починить. Никаких "чёрных ящиков".

---

## План урока

| # | Шаг | Что делаем |
|---|-----|------------|
| 1 | Подготовка окружения | Установка и импорт библиотек |
| 2 | Зачем нужна рекуррентность? | Последовательности - это не просто векторы |
| 3 | Простая RNN: формула | Как RNN обрабатывает последовательность |
| 4 | RNN своими руками на numpy | Реализация с нуля, шаг за шагом |
| 5 | Проблема исчезающего градиента | Визуализация: почему RNN "забывает" |
| 6 | LSTM: архитектура | Гейты: забывание, вход, выход, ячейка |
| 7 | LSTM своими руками на numpy | Каждый гейт - отдельно |
| 8 | GRU: упрощённая LSTM | Меньше параметров, похожий результат |
| 9 | RNN на PyTorch | nn.RNN, nn.LSTM, nn.GRU |
| 10 | Задача: классификация текста | Обучаем модель на реальных данных |
| 11 | Визуализация скрытых состояний | Что запоминает LSTM? |
| 12 | Сравнение RNN vs LSTM vs GRU | Графики и выводы |
| 13 | Практические задания | 5 заданий для самостоятельной работы |

---

---
## Шаг 1. Подготовка окружения

| Библиотека | Зачем |
|-----------|-------|
| **torch** | Создание и обучение RNN/LSTM |
| **numpy** | Реализация RNN с нуля (понимание механики) |
| **matplotlib** | Визуализация: графики, тепловые карты |
| **ipywidgets** | Интерактивные виджеты |

In [ ]:
# ============================================================
# ШАГ 1: Импортируем все нужные библиотеки
# ============================================================

import numpy as np                              # основная библиотека для массивов и математики
import matplotlib.pyplot as plt                 # для построения графиков и визуализаций
import torch                                    # PyTorch - основной фреймворк
import torch.nn as nn                           # нейросетевые слои (RNN, LSTM, Linear...)
import torch.optim as optim                     # оптимизаторы (SGD, Adam)
import torch.nn.functional as F                 # функции активации
from matplotlib.colors import LinearSegmentedColormap  # кастомные цветовые карты
from IPython.display import display, HTML       # красивый вывод в ноутбуке
import ipywidgets as widgets                    # интерактивные виджеты
from ipywidgets import interact, interactive, fixed  # декораторы для интерактива

# Настройка matplotlib
plt.rcParams['figure.figsize'] = (10, 6)        # размер графиков по умолчанию
plt.rcParams['font.size'] = 12                  # размер шрифта
plt.rcParams['axes.grid'] = True                # показываем сетку

# Фиксируем random seed для воспроизводимости
torch.manual_seed(42)                           # seed для PyTorch
np.random.seed(42)                              # seed для NumPy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # GPU или CPU
print(f"PyTorch: {torch.__version__}")
print(f"Устройство: {device}")
print("Все библиотеки импортированы!")

---
## Шаг 2. Зачем нужна рекуррентность?

Обычная нейросеть (MLP) принимает **фиксированный** вектор на вход.
Но что если данные - это **последовательность**?

- Текст: слова идут по порядку, смысл зависит от контекста
- Временной ряд: цена акций сегодня зависит от вчерашней
- Речь: звук в момент t зависит от предыдущих моментов

**Рекуррентная сеть (RNN)** обрабатывает последовательность **по шагам**,
передавая **скрытое состояние** (hidden state) от шага к шагу:

```
x1 -> [RNN] -> h1 -> [RNN] -> h2 -> [RNN] -> h3 -> ...
          ^              ^              ^
          |              |              |
         x0             x1             x2
```

Ключевая идея: **h_t = f(h_{t-1}, x_t)** - текущее состояние зависит от предыдущего и нового входа.

Это как читать книгу: вы понимаете каждое слово, потому что **помните** предыдущие!

In [ ]:
# ============================================================
# ШАГ 2: Демонстрация - почему последовательности важны
# ============================================================

# Пример: одинаковые слова, разный порядок - разный смысл
sentences = [
    ("Собака", "укусила", "человека"),              # собака агрессор
    ("Человек", "укусил", "собаку"),                # человек агрессор
]

print("Одинаковые слова, разный порядок:")
for i, words in enumerate(sentences):
    print(f"  {i+1}. {' '.join(words)}")

print()
print("MLP (без рекуррентности) видит только 'мешок слов':")
print("  Оба предложения -> одинаковый набор: {человек, собака, укусить}")
print("  MLP НЕ МОЖЕТ различить, КТО укусил КОГО!")
print()
print("RNN (с рекуррентностью) читает ПОСЛЕДОВАТЕЛЬНО:")
print("  Предложение 1: h0='Собака' -> h1='Собака кусает' -> h2='Собака кусает человека'")
print("  Предложение 2: h0='Человек' -> h1='Человек кусает' -> h2='Человек кусает собаку'")
print("  RNN МОЖЕТ различить порядок - скрытое состояние хранит контекст!")

---
## Шаг 3. Простая RNN: формула

**RNN** (Vanilla RNN / Elman RNN) обновляет скрытое состояние так:

```
h_t = tanh(W_hh @ h_{t-1} + W_xh @ x_t + b_h)
y_t = W_hy @ h_t + b_y
```

Где:
- **x_t** - вход в момент t (вектор размерности input_size)
- **h_{t-1}** - предыдущее скрытое состояние (вектор размерности hidden_size)
- **W_xh** - матрица весов входа (hidden_size x input_size)
- **W_hh** - матрица весов скрытого состояния (hidden_size x hidden_size)
- **b_h** - смещение (hidden_size)
- **tanh** - функция активации (сжимает в [-1, 1])
- **y_t** - выход в момент t

**Ключевое**: матрица W_hh - это то, что позволяет сети "помнить"!
Одни и те же веса применяются на каждом шаге.

In [ ]:
# ============================================================
# ШАГ 3: Визуализация архитектуры RNN
# ============================================================

fig, ax = plt.subplots(figsize=(14, 6))            # создаём фигуру

# Рисуем "развёрнутую" RNN (unrolled)
num_steps = 4                                      # количество шагов

for t in range(num_steps):
    x_pos = t * 4                                  # позиция по X

    # Вход x_t (снизу)
    ax.add_patch(plt.Rectangle((x_pos, 0), 1.5, 1, facecolor='#3498DB',  # синий
                               edgecolor='black', linewidth=2))
    ax.text(x_pos + 0.75, 0.5, f'$x_{t}$', ha='center', va='center',
            fontsize=14, fontweight='bold', color='white')

    # RNN-блок (посередине)
    ax.add_patch(plt.Rectangle((x_pos, 2), 1.5, 2, facecolor='#E74C3C',  # красный
                               edgecolor='black', linewidth=2))
    ax.text(x_pos + 0.75, 3, f'RNN', ha='center', va='center',
            fontsize=14, fontweight='bold', color='white')
    ax.text(x_pos + 0.75, 3.5, f'$h_{t}$', ha='center', va='center',
            fontsize=11, color='#FFEAA7')

    # Выход y_t (сверху)
    ax.add_patch(plt.Rectangle((x_pos, 5), 1.5, 1, facecolor='#2ECC71',  # зелёный
                               edgecolor='black', linewidth=2))
    ax.text(x_pos + 0.75, 5.5, f'$y_{t}$', ha='center', va='center',
            fontsize=14, fontweight='bold', color='white')

    # Стрелка x -> RNN
    ax.annotate('', xy=(x_pos + 0.75, 2), xytext=(x_pos + 0.75, 1),
                arrowprops=dict(arrowstyle='->', lw=2, color='#3498DB'))

    # Стрелка RNN -> y
    ax.annotate('', xy=(x_pos + 0.75, 5), xytext=(x_pos + 0.75, 4),
                arrowprops=dict(arrowstyle='->', lw=2, color='#2ECC71'))

    # Стрелка h_{t-1} -> h_t (рекуррентная связь)
    if t > 0:
        ax.annotate('', xy=(x_pos, 3), xytext=(x_pos - 1, 3),
                    arrowprops=dict(arrowstyle='->', lw=2.5, color='#F39C12',
                                   connectionstyle='arc3,rad=0'))

# Подписи
ax.text(-1.5, 0.5, 'Вход', fontsize=13, fontweight='bold', color='#3498DB', ha='center')
ax.text(-1.5, 3, 'Скрытое\nсостояние', fontsize=13, fontweight='bold', color='#E74C3C', ha='center')
ax.text(-1.5, 5.5, 'Выход', fontsize=13, fontweight='bold', color='#2ECC71', ha='center')

# Рекуррентная связь - подпись
ax.annotate('$W_{hh}$ (рекуррентная\nсвязь - память)', xy=(4, 3.5),
            xytext=(4, 4.5), fontsize=11, color='#F39C12', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#F39C12', lw=1.5))

ax.set_xlim(-2.5, num_steps * 4)
ax.set_ylim(-0.5, 7)
ax.set_title("Развёрнутая RNN: скрытое состояние передаётся от шага к шагу",  # заголовок
             fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print("Ключевой момент: оранжевые стрелки - это РЕКУРРЕНТНАЯ связь")
print("Одни и те же веса W_hh применяются на КАЖДОМ шаге")
print("Это позволяет сети 'запоминать' информацию из прошлого")

---
## Шаг 4. RNN своими руками на numpy

Реализуем RNN **полностью с нуля** - чтобы увидеть каждую операцию!

In [ ]:
# ============================================================
# ШАГ 4: Простая RNN на numpy - реализация с нуля
# ============================================================

class SimpleRNN:
    # Vanilla RNN, реализованная на numpy.
    #
    # Формула:
    #   h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)
    #   y_t = W_hy @ h_t + b_y
    #
    # Args:
    #   input_size: размерность входа x
    #   hidden_size: размерность скрытого состояния h
    #   output_size: размерность выхода y

    def __init__(self, input_size, hidden_size, output_size):
        # Инициализируем веса случайными числами (Xavier init)
        self.input_size = input_size                   # размер входа
        self.hidden_size = hidden_size                 # размер скрытого состояния
        self.output_size = output_size                 # размер выхода

        # Xavier initialization:方差 = 2/(fan_in + fan_out)
        scale_xh = np.sqrt(2.0 / (input_size + hidden_size))    # масштаб для W_xh
        scale_hh = np.sqrt(2.0 / (hidden_size + hidden_size))   # масштаб для W_hh
        scale_hy = np.sqrt(2.0 / (hidden_size + output_size))   # масштаб для W_hy

        np.random.seed(42)                             # фиксируем для воспроизводимости
        self.W_xh = np.random.randn(hidden_size, input_size) * scale_xh    # веса входа
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale_hh   # веса скрытого состояния
        self.b_h = np.zeros(hidden_size)               # смещение скрытого слоя
        self.W_hy = np.random.randn(output_size, hidden_size) * scale_hy   # веса выхода
        self.b_y = np.zeros(output_size)               # смещение выхода

    def forward(self, x_seq, h_0=None):
        # Прямой проход RNN по последовательности.
        #
        # Args:
        #   x_seq: список входов [x_0, x_1, ..., x_T] (каждый x_t - вектор)
        #   h_0: начальное скрытое состояние (если None - нули)
        # Returns:
        #   outputs: список выходов [y_0, y_1, ..., y_T]
        #   hidden_states: список скрытых состояний [h_0, h_1, ..., h_T]

        if h_0 is None:                                # если начальное состояние не задано
            h = np.zeros(self.hidden_size)              # начинаем с нулевого вектора
        else:
            h = h_0                                     # используем заданное

        outputs = []                                   # список выходов
        hidden_states = [h.copy()]                     # список скрытых состояний (включая h_0)

        for t, x_t in enumerate(x_seq):               # для каждого шага по времени
            # h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)
            h_pre = self.W_xh @ x_t + self.W_hh @ h + self.b_h  # линейное преобразование
            h = np.tanh(h_pre)                         # нелинейность tanh (сжимает в [-1, 1])

            # y_t = W_hy @ h_t + b_y
            y = self.W_hy @ h + self.b_y               # линейное преобразование выхода

            outputs.append(y)                          # сохраняем выход
            hidden_states.append(h.copy())             # сохраняем скрытое состояние

        return outputs, hidden_states                  # возвращаем выходы и скрытые состояния


# === Демонстрация ===

# Создаём RNN: вход=3, скрытое=5, выход=2
rnn = SimpleRNN(input_size=3, hidden_size=5, output_size=2)  # создаём RNN

# Создаём тестовую последовательность из 6 шагов
np.random.seed(100)                                    # фиксируем для примера
x_sequence = [np.random.randn(3) for _ in range(6)]   # 6 случайных входных векторов

# Прямой проход
outputs, hidden_states = rnn.forward(x_sequence)       # прогоняем через RNN

print("Прямой проход RNN:")
print(f"  Входная последовательность: {len(x_sequence)} шагов")
print(f"  Размер скрытого состояния: {rnn.hidden_size}")
print()
for t in range(len(outputs)):
    h = hidden_states[t+1]                              # скрытое состояние (t+1 т.к. h_0 в начале)
    print(f"  Шаг {t}: h_norm = {np.linalg.norm(h):.3f}, "
          f"h_mean = {h.mean():.3f}, y = [{outputs[t][0]:.3f}, {outputs[t][1]:.3f}]")

In [ ]:
# ============================================================
# Визуализация: как скрытое состояние меняется во времени
# ============================================================

# Собираем скрытые состояния в матрицу
h_matrix = np.array(hidden_states[1:])                # (T, hidden_size) - без h_0

fig, axes = plt.subplots(1, 2, figsize=(16, 5))       # два графика

# --- Heatmap скрытых состояний ---
ax = axes[0]
im = ax.imshow(h_matrix.T, cmap='RdBu_r', aspect='auto', interpolation='nearest')  # тепловая карта
ax.set_xlabel("Шаг времени t", fontsize=12)
ax.set_ylabel("Номер нейрона", fontsize=12)
ax.set_title("Скрытые состояния RNN: h[t] по шагам", fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Активация')

# Подписи шагов
for t in range(h_matrix.shape[0]):
    for n in range(h_matrix.shape[1]):
        val = h_matrix[t, n]
        if abs(val) > 0.3:                            # показываем только значимые
            ax.text(t, n, f'{val:.1f}', ha='center', va='center', fontsize=8,
                    color='white' if abs(val) > 0.6 else 'black')

# --- Норма скрытого состояния ---
ax = axes[1]
norms = [np.linalg.norm(h) for h in hidden_states[1:]]  # нормы h_t
means = [h.mean() for h in hidden_states[1:]]            # средние h_t
ax.plot(range(len(norms)), norms, 'o-', color='#E74C3C', linewidth=2, label='||h_t||')
ax.plot(range(len(means)), means, 's-', color='#3498DB', linewidth=2, label='mean(h_t)')
ax.set_xlabel("Шаг времени t", fontsize=12)
ax.set_ylabel("Значение", fontsize=12)
ax.set_title("Норма и среднее скрытого состояния", fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("Тепловая карта показывает, как активация нейронов меняется во времени")
print("Каждый столбец = один нейрон, каждая строка = один шаг времени")
print("Красный = положительная активация, синий = отрицательная")

---
## Шаг 5. Проблема исчезающего градиента

У простой RNN есть серьёзная проблема: **исчезающий градиент**.

Когда мы.backpropagate через много шагов, градиент умножается на W_hh на каждом шаге:
- Если eigenvalues W_hh < 1: градиент **экспоненциально затухает** -> сеть забывает
- Если eigenvalues W_hh > 1: градиент **экспоненциально растёт** -> взрыв (exploding gradient)

На практике tanh + инициализация приводят к **затуханию**.
RNN не может запоминать зависимости длиннее ~10-20 шагов!

In [ ]:
# ============================================================
# ШАГ 5: Визуализация проблемы исчезающего градиента
# ============================================================

# Симулируем затухание градиента через много шагов
# При backprop: dL/dh_0 = dL/dh_T * (W_hh^T)^T * ... * (W_hh^T)^T
# Норма градиента: ||grad|| ~ ||W_hh||^T * ||initial_grad||

np.random.seed(42)                                    # фиксируем seed
W_hh = np.random.randn(16, 16) * 0.5                  # типичная матрица W_hh RNN
eigenvalues = np.linalg.eigvals(W_hh)                 # собственные значения
spectral_radius = max(abs(eigenvalues))                # спектральный радиус

# Вычисляем норму градиента на каждом шаге (симуляция)
num_steps = 50                                        # количество шагов
initial_grad_norm = 1.0                               # начальная норма градиента

grad_norms = []                                       # нормы градиента на каждом шаге
grad_norm = initial_grad_norm                          # текущая норма
for t in range(num_steps):                            # для каждого шага
    grad_norms.append(grad_norm)                       # сохраняем
    # При backprop градиент умножается на W_hh
    grad_norm = grad_norm * spectral_radius            # затухание/рост

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- График нормы градиента ---
ax = axes[0]
ax.semilogy(range(num_steps), grad_norms, 'o-', color='#E74C3C', linewidth=2)
ax.axhline(y=1e-7, color='gray', linestyle='--', label='Порог забывания')
ax.set_xlabel("Шаг (расстояние от выхода)", fontsize=12)
ax.set_ylabel("Норма градиента (log scale)", fontsize=12)
ax.set_title(f"Исчезающий градиент (spectral radius = {spectral_radius:.3f})",  # заголовок
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.fill_between(range(num_steps), 1e-10, 1e-7, alpha=0.2, color='red', label='Зона забывания')

# --- Зависимость от spectral radius ---
ax = axes[1]
radii = [0.3, 0.5, 0.8, 1.0, 1.2]                   # разные spectral radii
colors_r = ['#E74C3C', '#F39C12', '#2ECC71', '#3498DB', '#9B59B6']
for radius, color in zip(radii, colors_r):            # для каждого радиуса
    norms = [initial_grad_norm * (radius ** t) for t in range(num_steps)]  # норма на каждом шаге
    ax.semilogy(range(num_steps), norms, '-', color=color, linewidth=2,
                label=f'r = {radius}')
ax.axhline(y=1e-7, color='gray', linestyle='--')
ax.set_xlabel("Шаг (расстояние от выхода)", fontsize=12)
ax.set_ylabel("Норма градиента (log scale)", fontsize=12)
ax.set_title("Затухание при разных spectral radii", fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("Ключевой вывод:")
print(f"  Spectral radius W_hh = {spectral_radius:.3f} < 1 -> градиент ЗАТУХАЕТ")
print("  После ~20 шагов градиент практически нулевой")
print("  -> RNN не может учить долгосрочные зависимости!")
print()
print("Решение: LSTM (Long Short-Term Memory) - использует гейты для контроля забывания")

---
## Шаг 6. LSTM: архитектура

**LSTM** (Hochreiter & Schmidhuber, 1997) решает проблему исчезающего градиента
с помощью **гейтов** (gate = вентиль) и **состояния ячейки** (cell state).

LSTM имеет **4 гейта**, каждый со своей матрицей весов:

| Гейт | Формула | Что делает |
|------|---------|------------|
| **Forget** (забывание) | f_t = sigmoid(W_f @ [h_{t-1}, x_t] + b_f) | Что забыть из C_{t-1}? |
| **Input** (вход) | i_t = sigmoid(W_i @ [h_{t-1}, x_t] + b_i) | Какую новую информацию добавить? |
| **Candidate** (кандидат) | g_t = tanh(W_g @ [h_{t-1}, x_t] + b_g) | Новая информация-кандидат |
| **Output** (выход) | o_t = sigmoid(W_o @ [h_{t-1}, x_t] + b_o) | Какую часть C_t вывести? |

Обновление:
```
C_t = f_t * C_{t-1} + i_t * g_t     # состояние ячейки: забыть старое + добавить новое
h_t = o_t * tanh(C_t)                 # скрытое состояние: фильтрованный выход
```

**Ключевая идея**: состояние ячейки C_t имеет **аддитивное** обновление
(а не мультипликативное как в RNN), поэтому градиент течёт без затухания!

In [ ]:
# ============================================================
# ШАГ 6: Визуализация архитектуры LSTM
# ============================================================

fig, ax = plt.subplots(figsize=(16, 10))            # создаём фигуру

# Основные блоки LSTM
blocks = {
    # (x, y, width, height, color, label)
    'forget': (2, 6, 4, 2, '#E74C3C', 'Forget Gate\n$f_t = \sigma(W_f \cdot [h,x] + b_f)$'),
    'input': (2, 3.5, 4, 2, '#2ECC71', 'Input Gate\n$i_t = \sigma(W_i \cdot [h,x] + b_i)$'),
    'candidate': (8, 3.5, 4, 2, '#F39C12', 'Candidate\n$g_t = \tanh(W_g \cdot [h,x] + b_g)$'),
    'output': (2, 1, 4, 2, '#3498DB', 'Output Gate\n$o_t = \sigma(W_o \cdot [h,x] + b_o)$'),
    'cell_update': (8, 6, 4, 2, '#9B59B6', 'Cell Update\n$C_t = f_t \cdot C_{t-1} + i_t \cdot g_t$'),
    'hidden': (8, 1, 4, 2, '#1ABC9C', 'Hidden State\n$h_t = o_t \cdot \tanh(C_t)$'),
}

for name, (x, y, w, h, color, label) in blocks.items():
    ax.add_patch(plt.Rectangle((x, y), w, h, facecolor=color,  # прямоугольник
                               edgecolor='black', linewidth=2, alpha=0.85))
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')

# Стрелки между блоками
arrows = [
    ((6, 7), (8, 7), 'f_t'),                           # forget -> cell_update
    ((6, 4.5), (8, 6.5), 'i_t'),                       # input -> cell_update
    ((10, 3.5), (10, 6), 'g_t'),                       # candidate -> cell_update
    ((6, 2), (8, 2), 'o_t'),                           # output -> hidden
    ((10, 6), (10, 3), 'tanh(C_t)'),                   # cell_update -> hidden
]

for (x1, y1), (x2, y2), label in arrows:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', lw=2, color='#2C3E50'))
    mx, my = (x1+x2)/2, (y1+y2)/2
    ax.text(mx + 0.3, my + 0.2, label, fontsize=9, color='#2C3E50', fontweight='bold')

# Подписи потока
ax.annotate('$C_{t-1}$ (из прошлого)', xy=(8, 8), xytext=(4, 9),
            fontsize=12, fontweight='bold', color='#9B59B6',
            arrowprops=dict(arrowstyle='->', color='#9B59B6', lw=2))
ax.annotate('$C_t$ (в будущее)', xy=(12, 7), xytext=(13, 8.5),
            fontsize=12, fontweight='bold', color='#9B59B6',
            arrowprops=dict(arrowstyle='->', color='#9B59B6', lw=2))
ax.annotate('$h_{t-1}$ + $x_t$', xy=(0, 4.5), xytext=(-2, 5.5),
            fontsize=12, fontweight='bold', color='#E67E22',
            arrowprops=dict(arrowstyle='->', color='#E67E22', lw=2))
ax.annotate('$h_t$ (выход)', xy=(12, 2), xytext=(13, 1),
            fontsize=12, fontweight='bold', color='#1ABC9C',
            arrowprops=dict(arrowstyle='->', color='#1ABC9C', lw=2))

ax.set_xlim(-3, 16)
ax.set_ylim(0, 10.5)
ax.set_title("Архитектура LSTM: 4 гейта + состояние ячейки", fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print("LSTM = 'конвейер' с 4 вентилями:")
print("  1. Forget: что ЗАБЫТЬ из старой памяти?")
print("  2. Input: какую НОВУЮ информацию добавить?")
print("  3. Candidate: чему РАВНА новая информация?")
print("  4. Output: какую часть памяти ВЫВЕСТИ наружу?")
print()
print("Ключевое: C_t = f_t*C_{t-1} + i_t*g_t")
print("  Это АДДИТИВНОЕ обновление -> градиент НЕ затухает!")

---
## Шаг 7. LSTM своими руками на numpy

Реализуем LSTM **полностью с нуля** - каждый гейт отдельно, каждую формулу!

In [ ]:
# ============================================================
# ШАГ 7: LSTM на numpy - реализация с нуля
# ============================================================

class SimpleLSTM:
    # LSTM (Long Short-Term Memory), реализованная на numpy.
    #
    # Формулы:
    #   f_t = sigmoid(W_f @ [h_{t-1}, x_t] + b_f)  -- forget gate
    #   i_t = sigmoid(W_i @ [h_{t-1}, x_t] + b_i)  -- input gate
    #   g_t = tanh(W_g @ [h_{t-1}, x_t] + b_g)     -- candidate
    #   o_t = sigmoid(W_o @ [h_{t-1}, x_t] + b_o)  -- output gate
    #   C_t = f_t * C_{t-1} + i_t * g_t             -- cell state
    #   h_t = o_t * tanh(C_t)                        -- hidden state
    #
    # Args:
    #   input_size: размерность входа x
    #   hidden_size: размерность скрытого состояния h и ячейки C

    def __init__(self, input_size, hidden_size):
        self.input_size = input_size                   # размер входа
        self.hidden_size = hidden_size                 # размер скрытого состояния

        # Объединённый размер [h_{t-1}, x_t]
        concat_size = hidden_size + input_size          # h + x вместе

        # Xavier initialization для всех матриц весов
        np.random.seed(42)                             # фиксируем для воспроизводимости
        scale = np.sqrt(2.0 / (concat_size + hidden_size))  # масштаб Xavier

        # Forget gate: что забыть
        self.W_f = np.random.randn(hidden_size, concat_size) * scale  # веса forget
        self.b_f = np.ones(hidden_size)                # ИНИЦИАЛИЗИРУЕМ 1 (по умолчанию НЕ забываем)

        # Input gate: что добавить
        self.W_i = np.random.randn(hidden_size, concat_size) * scale  # веса input
        self.b_i = np.zeros(hidden_size)               # смещение input

        # Candidate: новая информация
        self.W_g = np.random.randn(hidden_size, concat_size) * scale  # веса candidate
        self.b_g = np.zeros(hidden_size)               # смещение candidate

        # Output gate: что вывести
        self.W_o = np.random.randn(hidden_size, concat_size) * scale  # веса output
        self.b_o = np.zeros(hidden_size)               # смещение output

    def forward(self, x_seq, h_0=None, C_0=None):
        # Прямой проход LSTM по последовательности.
        #
        # Args:
        #   x_seq: список входов [x_0, x_1, ..., x_T]
        #   h_0: начальное скрытое состояние
        #   C_0: начальное состояние ячейки
        # Returns:
        #   outputs: список скрытых состояний [h_0, h_1, ..., h_T]
        #   cell_states: список состояний ячейки [C_0, C_1, ..., C_T]
        #   gate_values: словарь со значениями гейтов для визуализации

        if h_0 is None:
            h = np.zeros(self.hidden_size)              # нулевое скрытое состояние
        else:
            h = h_0
        if C_0 is None:
            C = np.zeros(self.hidden_size)              # нулевое состояние ячейки
        else:
            C = C_0

        outputs = [h.copy()]                           # список скрытых состояний
        cell_states = [C.copy()]                       # список состояний ячейки
        gate_values = {'f': [], 'i': [], 'g': [], 'o': []}  # значения гейтов

        for t, x_t in enumerate(x_seq):               # для каждого шага
            # Объединяем h_{t-1} и x_t
            concat = np.concatenate([h, x_t])          # [h_{t-1}; x_t]

            # Forget gate: что забыть из C_{t-1}
            f_t = self.sigmoid(self.W_f @ concat + self.b_f)  # sigmoid -> [0, 1]
            # f_t ~ 0 = забыть, f_t ~ 1 = сохранить

            # Input gate: какую новую информацию добавить
            i_t = self.sigmoid(self.W_i @ concat + self.b_i)  # sigmoid -> [0, 1]

            # Candidate: новая информация-кандидат
            g_t = np.tanh(self.W_g @ concat + self.b_g)       # tanh -> [-1, 1]

            # Обновление состояния ячейки: СТАРОЕ * forget + НОВОЕ * input
            C = f_t * C + i_t * g_t                    # АДДИТИВНОЕ обновление!

            # Output gate: какую часть C_t вывести
            o_t = self.sigmoid(self.W_o @ concat + self.b_o)  # sigmoid -> [0, 1]

            # Скрытое состояние: фильтрованный выход из ячейки
            h = o_t * np.tanh(C)                       # выходной гейт фильтрует tanh(C)

            # Сохраняем всё для визуализации
            outputs.append(h.copy())
            cell_states.append(C.copy())
            gate_values['f'].append(f_t.copy())
            gate_values['i'].append(i_t.copy())
            gate_values['g'].append(g_t.copy())
            gate_values['o'].append(o_t.copy())

        return outputs, cell_states, gate_values

    @staticmethod
    def sigmoid(x):
        # Сигмоида: sigma(x) = 1 / (1 + exp(-x))
        return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))  # clip для стабильности


# === Демонстрация ===
lstm = SimpleLSTM(input_size=3, hidden_size=8)        # создаём LSTM

# Тестовая последовательность
x_seq = [np.random.randn(3) for _ in range(10)]       # 10 шагов

# Прямой проход
outputs, cell_states, gate_values = lstm.forward(x_seq)  # прогоняем через LSTM

print(f"LSTM: input_size=3, hidden_size=8, 10 шагов")
print(f"Скрытые состояния: {len(outputs)} (включая h_0)")
print(f"Состояния ячейки: {len(cell_states)} (включая C_0)")
print(f"Значения гейтов: {len(gate_values['f'])} шагов")
print()
print("LSTM работает! Теперь визуализируем гейты...")

In [ ]:
# ============================================================
# Визуализация гейтов LSTM
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))      # 2x2 сетка

gate_names = {'f': 'Forget Gate (что забыть)', 'i': 'Input Gate (что добавить)',
              'g': 'Candidate (новая инф.)', 'o': 'Output Gate (что вывести)'}

for idx, (gate_key, gate_name) in enumerate(gate_names.items()):
    ax = axes[idx // 2][idx % 2]                       # выбираем ось
    gate_matrix = np.array(gate_values[gate_key])      # (T, hidden_size)
    im = ax.imshow(gate_matrix.T, cmap='RdYlGn' if gate_key != 'g' else 'RdBu_r',
                   aspect='auto', interpolation='nearest', vmin=-1 if gate_key == 'g' else 0, vmax=1)
    ax.set_xlabel("Шаг времени t", fontsize=11)
    ax.set_ylabel("Номер нейрона", fontsize=11)
    ax.set_title(gate_name, fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("Значения гейтов LSTM на каждом шаге", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("Зелёный = 1 (пропустить), Красный = 0 (заблокировать)")
print("Forget gate: 1 = сохранить, 0 = забыть")
print("Input gate: 1 = добавить, 0 = проигнорировать")
print("Output gate: 1 = вывести, 0 = скрыть")
print()
print("Гейты позволяют LSTM 'выбирать', что помнить, а что забыть!")

---
## Шаг 8. GRU: упрощённая LSTM

**GRU** (Cho et al., 2014) - упрощённая версия LSTM с двумя гейтами вместо четырёх:

| Гейт | Формула | Что делает |
|------|---------|------------|
| **Reset** | r_t = sigmoid(W_r @ [h_{t-1}, x_t]) | Какую часть прошлого игнорировать? |
| **Update** | z_t = sigmoid(W_z @ [h_{t-1}, x_t]) | Баланс между старым и новым |

Обновление:
```
h_t = (1 - z_t) * h_{t-1} + z_t * tanh(W_h @ [r_t * h_{t-1}, x_t])
```

**Сравнение LSTM vs GRU:**
- LSTM: 4 гейта, 2 состояния (h и C) - мощнее для длинных зависимостей
- GRU: 2 гейта, 1 состояние (h) - быстрее обучается, меньше параметров

In [ ]:
# ============================================================
# ШАГ 8: GRU на numpy - реализация с нуля
# ============================================================

class SimpleGRU:
    # GRU (Gated Recurrent Unit), реализованная на numpy.
    #
    # Формулы:
    #   z_t = sigmoid(W_z @ [h_{t-1}, x_t] + b_z)  -- update gate
    #   r_t = sigmoid(W_r @ [h_{t-1}, x_t] + b_r)  -- reset gate
    #   h_tilde = tanh(W_h @ [r_t * h_{t-1}, x_t] + b_h)  -- candidate
    #   h_t = (1 - z_t) * h_{t-1} + z_t * h_tilde  -- update
    #
    # Args:
    #   input_size: размерность входа
    #   hidden_size: размерность скрытого состояния

    def __init__(self, input_size, hidden_size):
        self.input_size = input_size
        self.hidden_size = hidden_size
        concat_size = hidden_size + input_size

        np.random.seed(42)
        scale = np.sqrt(2.0 / (concat_size + hidden_size))

        # Update gate: баланс старого и нового
        self.W_z = np.random.randn(hidden_size, concat_size) * scale
        self.b_z = np.zeros(hidden_size)

        # Reset gate: какую часть прошлого игнорировать
        self.W_r = np.random.randn(hidden_size, concat_size) * scale
        self.b_r = np.zeros(hidden_size)

        # Candidate: новое скрытое состояние
        self.W_h = np.random.randn(hidden_size, concat_size) * scale
        self.b_h = np.zeros(hidden_size)

    def forward(self, x_seq, h_0=None):
        # Прямой проход GRU

        if h_0 is None:
            h = np.zeros(self.hidden_size)
        else:
            h = h_0

        outputs = [h.copy()]
        gate_values = {'z': [], 'r': []}

        for t, x_t in enumerate(x_seq):
            concat = np.concatenate([h, x_t])

            # Update gate
            z_t = 1.0 / (1.0 + np.exp(-(self.W_z @ concat + self.b_z)))  # sigmoid

            # Reset gate
            r_t = 1.0 / (1.0 + np.exp(-(self.W_r @ concat + self.b_r)))  # sigmoid

            # Candidate (reset gate модулирует h_{t-1})
            concat_r = np.concatenate([r_t * h, x_t])  # reset gate "обнуляет" часть h
            h_tilde = np.tanh(self.W_h @ concat_r + self.b_h)  # candidate

            # Update: баланс между старым и новым
            h = (1 - z_t) * h + z_t * h_tilde          # интерполяция

            outputs.append(h.copy())
            gate_values['z'].append(z_t.copy())
            gate_values['r'].append(r_t.copy())

        return outputs, gate_values


# Демонстрация
gru = SimpleGRU(input_size=3, hidden_size=8)           # создаём GRU
gru_outputs, gru_gates = gru.forward(x_seq)            # прогоняем

print(f"GRU: input_size=3, hidden_size=8, 10 шагов")
print(f"Скрытые состояния: {len(gru_outputs)}")
print()
print("Сравнение параметров:")
print(f"  LSTM: {sum(p.size for p in [lstm.W_f, lstm.W_i, lstm.W_g, lstm.W_o])} весов + смещения")
print(f"  GRU:  {sum(p.size for p in [gru.W_z, gru.W_r, gru.W_h])} весов + смещения")
print(f"  GRU на ~25% меньше параметров!")

---
## Шаг 9. RNN на PyTorch

PyTorch предоставляет готовые реализации RNN, LSTM и GRU.
Теперь, когда мы понимаем, как они работают изнутри,
использовать их будет осознанно!

In [ ]:
# ============================================================
# ШАГ 9: nn.RNN, nn.LSTM, nn.GRU в PyTorch
# ============================================================

# --- nn.RNN ---
rnn_torch = nn.RNN(input_size=10, hidden_size=20,     # простая RNN
                    num_layers=1, batch_first=True)    # batch_first: (batch, seq, features)
print("nn.RNN параметры:")
for name, param in rnn_torch.named_parameters():
    print(f"  {name}: {param.shape}")                  # размеры весов

# Прямой проход
x_dummy = torch.randn(2, 5, 10)                       # (batch=2, seq_len=5, input_size=10)
output_rnn, h_n_rnn = rnn_torch(x_dummy)              # прогоняем
print(f"\nRNN: вход {x_dummy.shape} -> выход {output_rnn.shape}, h_n {h_n_rnn.shape}")

# --- nn.LSTM ---
lstm_torch = nn.LSTM(input_size=10, hidden_size=20,   # LSTM
                      num_layers=1, batch_first=True)
x_dummy = torch.randn(2, 5, 10)                       # тот же вход
output_lstm, (h_n_lstm, c_n_lstm) = lstm_torch(x_dummy)  # LSTM возвращает (h, C)
print(f"\nLSTM: вход {x_dummy.shape} -> выход {output_lstm.shape}")
print(f"  h_n {h_n_lstm.shape}, c_n {c_n_lstm.shape}")

# --- nn.GRU ---
gru_torch = nn.GRU(input_size=10, hidden_size=20,     # GRU
                    num_layers=1, batch_first=True)
output_gru, h_n_gru = gru_torch(x_dummy)
print(f"\nGRU: вход {x_dummy.shape} -> выход {output_gru.shape}, h_n {h_n_gru.shape}")

print()
print("Все три возвращают:")
print("  output: (batch, seq_len, hidden_size) - скрытые состояния на КАЖДОМ шаге")
print("  h_n: (num_layers, batch, hidden_size) - скрытое состояние ПОСЛЕДНЕГО шага")
print("  LSTM дополнительно: c_n - состояние ячейки последнего шага")

---
## Шаг 10. Задача: классификация текста

Обучим LSTM для **анализа тональности** отзывов.
Для простоты создадим синтетический датасет с позитивными и негативными отзывами.

In [ ]:
# ============================================================
# ШАГ 10a: Создаём датасет для анализа тональности
# ============================================================

# Синтетический датасет: позитивные vs негативные отзывы
positive_reviews = [
    "этот фильм отличный и замечательный",
    "прекрасная работа актёров и режиссёра",
    "мне очень понравилась эта книга",
    "замечательный день и прекрасная погода",
    "отличный сервис и вкусная еда",
    "я счастлив что посмотрел этот фильм",
    "великолепная музыка и красивые кадры",
    "это лучшее что я видел в этом году",
    "прекрасное выступление и овации",
    "радость и восторг от просмотра",
    "удивительно красивый и трогательный",
    "потрясающая игра актёров",
    "я доволен покупкой это супер",
    "восхитительный вкус и аромат",
    "блестящая идея и отличное исполнение",
]

negative_reviews = [
    "этот фильм ужасный и скучный",
    "плохая работа актёров и режиссёра",
    "мне не понравилась эта книга",
    "ужасный день и отвратительная погода",
    "плохой сервис и невкусная еда",
    "я разочарован что посмотрел этот фильм",
    "скучная музыка и уродливые кадры",
    "это худшее что я видел в этом году",
    "провальное выступление и свист",
    "грусть и разочарование от просмотра",
    "удивительно скучный и унылый",
    "отвратительная игра актёров",
    "я недоволен покупкой это кошмар",
    "мерзкий вкус и запах",
    "ужасная идея и плохое исполнение",
]

# Объединяем и создаём метки
all_reviews = positive_reviews + negative_reviews       # все отзывы
labels = [1] * len(positive_reviews) + [0] * len(negative_reviews)  # 1=позитив, 0=негатив

print(f"Всего отзывов: {len(all_reviews)}")
print(f"  Позитивных: {len(positive_reviews)}")
print(f"  Негативных: {len(negative_reviews)}")
print()
print("Примеры:")
for i in [0, 1, 15, 16]:
    label_str = "ПОЗИТИВ" if labels[i] == 1 else "НЕГАТИВ"
    print(f"  [{label_str}] {all_reviews[i]}")

In [ ]:
# ============================================================
# ШАГ 10b: Токенизация и создание словаря
# ============================================================

import re

def tokenize(text):
    # Простая токенизация: нижний регистр + убираем пунктуацию + split
    text = text.lower()                                # нижний регистр
    text = re.sub(r'[^\w\s]', '', text)              # убираем пунктуацию
    return text.split()                                # разбиваем по пробелам

# Строим словарь по всем отзывам
word_counts = Counter()                                # счётчик частот
for review in all_reviews:
    tokens = tokenize(review)                          # токенизируем
    word_counts.update(tokens)                         # обновляем частоты

# Добавляем спецтокены
vocab = {'<PAD>': 0, '<UNK>': 1}                      # PAD = заполнитель, UNK = неизвестное слово
for word, count in word_counts.most_common():          # по убыванию частоты
    vocab[word] = len(vocab)                           # назначаем индексы

vocab_size = len(vocab)                                # размер словаря
print(f"Размер словаря: {vocab_size} слов")
print(f"Первые 10 слов: {list(vocab.items())[:10]}")

# Функция для кодирования отзыва в последовательность индексов
def encode_review(text, vocab, max_len=10):
    # Кодируем текст в последовательность индексов
    # Если длиннее max_len - обрезаем, если короче - дополняем PAD
    tokens = tokenize(text)                            # токенизируем
    indices = [vocab.get(t, vocab['<UNK>']) for t in tokens]  # индексы
    # Обрезаем или дополняем
    if len(indices) > max_len:                         # если слишком длинный
        indices = indices[:max_len]                    # обрезаем
    else:                                              # если короткий
        indices = indices + [vocab['<PAD>']] * (max_len - len(indices))  # дополняем PAD
    return indices

# Кодируем все отзывы
MAX_LEN = 10                                           # максимальная длина последовательности
X = torch.tensor([encode_review(r, vocab, MAX_LEN) for r in all_reviews],  # матрица индексов
                 dtype=torch.long)
y = torch.tensor(labels, dtype=torch.float32)          # метки (0 или 1)

print(f"\nX shape: {X.shape}  (отзывы x длина)")
print(f"y shape: {y.shape}  (метки)")
print(f"\nПример закодированного отзыва:")
print(f"  Текст: '{all_reviews[0]}'")
print(f"  Индексы: {X[0].tolist()}")
print(f"  Метка: {y[0].item()} ({'позитив' if y[0] else 'негатив'})")

In [ ]:
# ============================================================
# ШАГ 10c: Модель LSTM для классификации
# ============================================================

class SentimentLSTM(nn.Module):
    # LSTM-модель для анализа тональности текста.
    #
    # Архитектура:
    #   1. Embedding: индексы слов -> плотные векторы
    #   2. LSTM: обрабатывает последовательность эмбеддингов
    #   3. Linear: скрытое состояние -> вероятность позитив/негатив
    #
    # Args:
    #   vocab_size: размер словаря
    #   embed_dim: размерность эмбеддинга
    #   hidden_dim: размерность скрытого состояния LSTM
    #   num_layers: количество слоёв LSTM

    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32, num_layers=1):
        super().__init__()                             # инициализируем nn.Module
        self.hidden_dim = hidden_dim                   # размер скрытого состояния
        self.num_layers = num_layers                   # количество слоёв

        # Слой эмбеддингов: превращает индексы слов в плотные векторы
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)  # PAD -> нулевой вектор

        # LSTM: обрабатывает последовательность
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=0.1 if num_layers > 1 else 0)

        # Полносвязный слой: hidden_dim -> 1 (вероятность позитив)
        self.fc = nn.Linear(hidden_dim, 1)

        # Sigmoid: превращает логит в вероятность [0, 1]
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Прямой проход модели.
        #
        # Args:
        #   x: (batch, seq_len) - последовательности индексов слов
        # Returns:
        #   (batch, 1) - вероятности позитивного класса

        batch_size = x.size(0)                         # размер батча

        # 1. Эмбеддинги: (batch, seq_len) -> (batch, seq_len, embed_dim)
        embedded = self.embedding(x)                   # превращаем индексы в векторы

        # 2. LSTM: (batch, seq_len, embed_dim) -> (batch, seq_len, hidden_dim)
        lstm_out, (h_n, c_n) = self.lstm(embedded)    # обрабатываем последовательность

        # 3. Берём скрытое состояние ПОСЛЕДНЕГО шага
        # h_n: (num_layers, batch, hidden_dim) -> (batch, hidden_dim)
        last_hidden = h_n[-1]                          # скрытое состояние последнего слоя

        # 4. Полносвязный слой: (batch, hidden_dim) -> (batch, 1)
        logits = self.fc(last_hidden)                  # логит (до sigmoid)

        # 5. Sigmoid: логит -> вероятность
        probs = self.sigmoid(logits)                   # вероятность [0, 1]

        return probs


# Создаём модель
model = SentimentLSTM(vocab_size=vocab_size, embed_dim=16, hidden_dim=32)
print("Модель SentimentLSTM:")
print(model)
print()
total_params = sum(p.numel() for p in model.parameters())  # количество параметров
print(f"Всего параметров: {total_params:,}")

In [ ]:
# ============================================================
# ШАГ 10d: Обучаем модель
# ============================================================

# Функция потерь: бинарная кросс-энтропия
criterion = nn.BCELoss()                               # Binary Cross-Entropy Loss

# Оптимизатор: Adam
optimizer = optim.Adam(model.parameters(), lr=0.01)   # Adam с lr=0.01

# Обучаем
num_epochs = 50                                        # количество эпох
losses = []                                            # для графика loss
accuracies = []                                        # для графика accuracy

for epoch in range(num_epochs):
    model.train()                                      # режим обучения

    # Прямой проход
    outputs = model(X).squeeze()                       # (batch,) - вероятности
    loss = criterion(outputs, y)                       # вычисляем loss

    # Обратный проход
    optimizer.zero_grad()                              # обнуляем градиенты
    loss.backward()                                    # backpropagation
    optimizer.step()                                   # обновляем веса

    # Вычисляем accuracy
    predictions = (outputs >= 0.5).float()             # порог 0.5
    correct = (predictions == y).sum().item()          # количество правильных
    accuracy = correct / len(y)                        # доля правильных

    losses.append(loss.item())                         # сохраняем loss
    accuracies.append(accuracy)                        # сохраняем accuracy

    if (epoch + 1) % 10 == 0:                         # каждые 10 эпох
        print(f"Эпоха {epoch+1:3d}: loss = {loss.item():.4f}, accuracy = {accuracy:.2%}")

print()
print("Обучение завершено!")

# Визуализация
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, num_epochs+1), losses, color='#E74C3C', linewidth=2)
ax1.set_xlabel("Эпоха", fontsize=12)
ax1.set_ylabel("Loss", fontsize=12)
ax1.set_title("Loss при обучении LSTM", fontsize=13, fontweight='bold')

ax2.plot(range(1, num_epochs+1), accuracies, color='#2ECC71', linewidth=2)
ax2.set_xlabel("Эпоха", fontsize=12)
ax2.set_ylabel("Accuracy", fontsize=12)
ax2.set_title("Accuracy при обучении LSTM", fontsize=13, fontweight='bold')
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

---
## Шаг 11. Визуализация скрытых состояний

Посмотрим, **что** LSTM запоминает на каждом шаге при обработке текста!

In [ ]:
# ============================================================
# ШАГ 11: Визуализация скрытых состояний LSTM
# ============================================================

model.eval()                                           # режим оценки (без dropout)

# Берём два отзыва: позитивный и негативный
pos_idx = 0                                            # первый позитивный
neg_idx = len(positive_reviews)                        # первый негативный

with torch.no_grad():                                  # без вычисления градиентов
    for idx, label in [(pos_idx, "ПОЗИТИВ"), (neg_idx, "НЕГАТИВ")]:
        x_single = X[idx:idx+1]                        # (1, seq_len)
        embedded = model.embedding(x_single)           # (1, seq_len, embed_dim)
        lstm_out, (h_n, c_n) = model.lstm(embedded)   # (1, seq_len, hidden_dim)

        # lstm_out[0] - скрытые состояния на каждом шаге
        h_states = lstm_out[0].numpy()                 # (seq_len, hidden_dim)

        # Декодируем индексы обратно в слова
        idx2word = {v: k for k, v in vocab.items()}    # обратный словарь
        words = [idx2word.get(idx.item(), '<PAD>') for idx in x_single[0]]

        # Визуализация heatmap
        fig, ax = plt.subplots(figsize=(14, 5))
        im = ax.imshow(h_states.T, cmap='RdBu_r', aspect='auto',
                       interpolation='nearest', vmin=-1, vmax=1)
        ax.set_xticks(range(len(words)))
        ax.set_xticklabels(words, rotation=45, ha='right', fontsize=11)
        ax.set_ylabel("Номер нейрона", fontsize=12)
        ax.set_title(f"Скрытые состояния LSTM: {label} отзыв",
                     fontsize=13, fontweight='bold')
        plt.colorbar(im, ax=ax, label='Активация')
        plt.tight_layout()
        plt.show()

print("Сравните: для ПОЗИТИВНОГО отзыва активации отличаются от НЕГАТИВНОГО")
print("LSTM 'запоминает' ключевые слова и их контекст!")

---
## Шаг 12. Сравнение RNN vs LSTM vs GRU

Сравним три архитектуры на нашей задаче классификации!

In [ ]:
# ============================================================
# ШАГ 12: Сравнение RNN vs LSTM vs GRU на задаче классификации
# ============================================================

class SentimentRNN(nn.Module):
    # Vanilla RNN для классификации (аналог SentimentLSTM, но с nn.RNN)
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        rnn_out, h_n = self.rnn(embedded)
        logits = self.fc(h_n[-1])
        return self.sigmoid(logits)


class SentimentGRU(nn.Module):
    # GRU для классификации
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        gru_out, h_n = self.gru(embedded)
        logits = self.fc(h_n[-1])
        return self.sigmoid(logits)


# Обучаем все три модели и сравниваем
models = {
    'RNN': SentimentRNN(vocab_size),
    'LSTM': SentimentLSTM(vocab_size),
    'GRU': SentimentGRU(vocab_size)
}

results = {}                                           # результаты обучения

for name, model_cls in models.items():
    # Пересоздаём модель для чистоты эксперимента
    if name == 'RNN':
        m = SentimentRNN(vocab_size)
    elif name == 'LSTM':
        m = SentimentLSTM(vocab_size)
    else:
        m = SentimentGRU(vocab_size)

    criterion = nn.BCELoss()
    optimizer = optim.Adam(m.parameters(), lr=0.01)

    epoch_losses = []                                  # loss по эпохам
    epoch_accs = []                                    # accuracy по эпохам

    for epoch in range(50):
        m.train()
        outputs = m(X).squeeze()
        loss = criterion(outputs, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        predictions = (outputs >= 0.5).float()
        accuracy = (predictions == y).float().mean().item()

        epoch_losses.append(loss.item())
        epoch_accs.append(accuracy)

    results[name] = {'losses': epoch_losses, 'accuracies': epoch_accs}
    final_acc = epoch_accs[-1]
    print(f"{name:5s}: финальная accuracy = {final_acc:.2%}, параметров = {sum(p.numel() for p in m.parameters()):,}")

# Визуализация сравнения
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = {'RNN': '#E74C3C', 'LSTM': '#2ECC71', 'GRU': '#3498DB'}

for name, data in results.items():
    ax1.plot(range(1, 51), data['losses'], color=colors[name], linewidth=2, label=name)
    ax2.plot(range(1, 51), data['accuracies'], color=colors[name], linewidth=2, label=name)

ax1.set_xlabel("Эпоха", fontsize=12)
ax1.set_ylabel("Loss", fontsize=12)
ax1.set_title("Loss: RNN vs LSTM vs GRU", fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)

ax2.set_xlabel("Эпоха", fontsize=12)
ax2.set_ylabel("Accuracy", fontsize=12)
ax2.set_title("Accuracy: RNN vs LSTM vs GRU", fontsize=13, fontweight='bold')
ax2.set_ylim(0, 1.05)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("Выводы:")
print("  1. LSTM и GRU обычно обучаются быстрее и стабильнее RNN")
print("  2. GRU чуть быстрее LSTM (меньше параметров)")
print("  3. На коротких последовательностях разница может быть небольшой")
print("  4. На длинных последовательностях LSTM/GRU значительно превосходят RNN")

---
## Шаг 13. Практические задания

Теперь ваша очередь! Вот 5 заданий для закрепления материала.

> Помните: вы - автор. Меняйте код, ломайте, чините. Это ваш ноутбук!

In [ ]:
# ============================================================
# ЗАДАНИЕ 1: Двунаправленная LSTM (BiLSTM)
# ============================================================

# BiLSTM обрабатывает последовательность В ОБЕ СТОРОНЫ:
#   - Прямой LSTM: слева направо
#   - Обратный LSTM: справа налево
#   - Выход: конкатенация обоих
#
# TODO: Создайте модель SentimentBiLSTM на основе nn.LSTM с bidirectional=True
# Подсказка: hidden_dim выхода будет 2*hidden_dim (конкатенация двух направлений)

class SentimentBiLSTM(nn.Module):
    # Двунаправленная LSTM для классификации
    #
    # TODO: Допишите модель!
    # Подсказка:
    #   - nn.LSTM(..., bidirectional=True)
    #   - Выход LSTM: (batch, seq_len, 2*hidden_dim)
    #   - h_n: (2*num_layers, batch, hidden_dim)
    #   - Для fc: берём h_n[-2:] и объединяем

    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32):
        super().__init__()
        # Ваш код здесь:
        pass

    def forward(self, x):
        # Ваш код здесь:
        pass

print("ЗАДАНИЕ: реализуйте SentimentBiLSTM")
print("   Подсказка: bidirectional=True в nn.LSTM")

In [ ]:
# ============================================================
# ЗАДАНИЕ 2: Генерация текста с LSTM
# ============================================================

# LSTM можно использовать для ГЕНЕРАЦИИ текста!
# Идея: обучаем LSTM предсказывать СЛЕДУЮЩИЙ символ/слово
# Генерация: подаём начальный текст и семплируем следующие токены
#
# TODO: Создайте модель для генерации текста посимвольно
# Подсказка:
#   1. Создайте словарь символов из обучающего текста
#   2. Обучите LSTM предсказывать следующий символ
#   3. Для генерации: feed forward + sample из распределения

print("ЗАДАНИЕ: создайте LSTM-модель для генерации текста")
print("   Шаг 1: подготовьте текст и словарь символов")
print("   Шаг 2: обучите LSTM предсказывать следующий символ")
print("   Шаг 3: реализуйте функцию generate(model, start_text, length)")
print()
print("Пример генерации:")
print("  Вход: 'кот'")
print("  Выход: 'кот сидит на коврике и мурлычет'")

In [ ]:
# ============================================================
# ЗАДАНИЕ 3: Исследуйте влияние hidden_size
# ============================================================

# Как размер скрытого состояния влияет на обучение?
# Попробуйте hidden_dim = 8, 16, 32, 64, 128
# Постройте график: hidden_dim vs финальная accuracy

# Ваш код здесь:
hidden_sizes = [8, 16, 32, 64, 128]                   # размеры для эксперимента

print("ЗАДАНИЕ: обучите LSTM с разными hidden_size и сравните")
print("   Постройте график: hidden_dim vs accuracy")
print("   Есть ли точка, после которой увеличение не помогает?")

In [ ]:
# ============================================================
# ЗАДАНИЕ 4: Визуализируйте состояние ячейки LSTM
# ============================================================

# В шаге 11 мы визуализировали скрытое состояние h_t
# Теперь визуализируйте СОСТОЯНИЕ ЯЧЕЙКИ C_t
# Как C_t меняется при обработке позитивного vs негативного отзыва?

# Подсказка: модифицируйте SentimentLSTM так, чтобы он возвращал
# все состояния ячейки, а не только скрытые состояния

print("ЗАДАНИЕ: визуализируйте C_t (cell state) LSTM")
print("   Сравните C_t для позитивного и негативного отзыва")
print("   Как C_t 'хранит' информацию на протяжении последовательности?")

In [ ]:
# ============================================================
# ЗАДАНИЕ 5: Сравните на длинных последовательностях
# ============================================================

# На коротких текстах RNN и LSTM работают похоже.
# Но на ДЛИННЫХ последовательностях LSTM значительно лучше!
#
# TODO: Создайте датасет с длинными последовательностями (50+ токенов)
# и сравните RNN vs LSTM vs GRU.
#
# Подсказка: используйте статьи или длинные отзывы

print("ЗАДАНИЕ: сравните RNN vs LSTM vs GRU на длинных последовательностях")
print("   Создайте датасет с seq_len >= 50")
print("   Обучите все три модели и сравните accuracy")
print()
print("=" * 60)
print("Поздравляем! Вы прошли блокнот 'Рекуррентные сети (RNN/LSTM)'")
print("=" * 60)
print()
print("Что вы освоили:")
print("  + Простая RNN: формула и реализация с нуля")
print("  + Проблема исчезающего градиента (визуально)")
print("  + LSTM: 4 гейта и состояние ячейки")
print("  + GRU: 2 гейта, упрощённая архитектура")
print("  + RNN/LSTM/GRU в PyTorch")
print("  + Классификация текста с LSTM")
print("  + Визуализация скрытых состояний")
print()
print("Следующий блокнот: 07_attention_transformer.ipynb -> Attention и Transformer")